# 02. 응용: 편집 깊이와 context window

목표: context-dependent toy watermark를 만든 뒤 token substitution 비율을 높이며 right/wrong key의 z-score와 원문 window 생존율을 비교한다.

In [ ]:
import hashlib, hmac, math, random

VOCAB = 'analysis evidence model token context signal robust clear useful method result system'.split()
GAMMA, CONTEXT = 0.5, 3

def green_set(key: bytes, prefix: tuple[str, ...]) -> set[str]:
    payload = ' '.join(prefix).encode()
    ranked = sorted((hmac.new(key, payload + b'|' + word.encode(), hashlib.sha256).digest(), word) for word in VOCAB)
    return {word for _, word in ranked[: round(len(VOCAB) * GAMMA)]}

def generate(key: bytes, n: int, delta: float, seed: int) -> list[str]:
    rng, out = random.Random(seed), []
    for _ in range(n):
        greens = green_set(key, tuple(out[-CONTEXT:]))
        weights = [math.exp(delta if word in greens else 0.0) for word in VOCAB]
        out.append(rng.choices(VOCAB, weights=weights, k=1)[0])
    return out

def z_score(tokens: list[str], key: bytes) -> float:
    green = sum(token in green_set(key, tuple(tokens[max(0, i-CONTEXT):i])) for i, token in enumerate(tokens))
    n = len(tokens)
    return (green - GAMMA*n) / math.sqrt(n*GAMMA*(1-GAMMA))

def edit(tokens: list[str], rate: float, seed: int) -> list[str]:
    rng, edited = random.Random(seed), tokens.copy()
    for i, old in enumerate(edited):
        if rng.random() < rate:
            edited[i] = rng.choice([word for word in VOCAB if word != old])
    return edited

def surviving_windows(original: list[str], edited: list[str], width: int = CONTEXT + 1) -> float:
    total = max(1, len(original) - width + 1)
    same = sum(original[i:i+width] == edited[i:i+width] for i in range(total))
    return same / total

In [ ]:
key = b'editing-lab-key'
original = generate(key, n=1500, delta=1.0, seed=21)
print('edit rate | surviving windows | z(right) | z(wrong)')
print('-' * 56)
for rate in [0.00, 0.02, 0.05, 0.10, 0.20, 0.40, 0.70, 1.00]:
    revised = edit(original, rate, seed=99)
    surviving = surviving_windows(original, revised)
    print(f'{rate:8.0%} | {surviving:17.1%} | {z_score(revised, key):8.2f} | {z_score(revised, b"wrong"):8.2f}')

## 부분 혼합 실험

marked passage 일부와 independently generated plain passage를 섞으면 전체 score가 어떻게 희석되는지 확인한다.

In [ ]:
plain = generate(key, n=1500, delta=0.0, seed=88)
for marked_share in [1.0, 0.75, 0.50, 0.25, 0.0]:
    cut = round(len(original) * marked_share)
    mixed = original[:cut] + plain[cut:]
    print(f'marked share={marked_share:4.0%}, z={z_score(mixed, key):6.2f}')

## 관찰 포인트

token substitution은 실제 paraphrase의 불완전한 proxy다. 실제 edit는 의미·문장 길이·tokenization을 함께 바꾼다. 그래도 context window 하나가 깨질 때 주변 evidence도 바뀌는 구조와 marked/unmarked mixture의 희석을 확인할 수 있다. 이 결과를 특정 provider watermark의 우회 성능으로 해석하지 않는다.